In [1]:
import pandas as pd
from pathlib import Path

In [5]:
notebook_dir = Path.cwd()  

data_path = notebook_dir / "../data/raw/sales_records.csv"

data_path = data_path.resolve()
print("Reading from:", data_path)

df = pd.read_csv(data_path)

df.head()


Reading from: C:\Users\Marizze Julliane\OneDrive\Desktop\Mark\DashIQ\data\raw\sales_records.csv


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Sub-Saharan Africa,South Africa,Fruits,Offline,M,7/27/2012,443368995,7/28/2012,1593,9.33,6.92,14862.69,11023.56,3839.13
1,Middle East and North Africa,Morocco,Clothes,Online,M,9/14/2013,667593514,10/19/2013,4611,109.28,35.84,503890.08,165258.24,338631.84
2,Australia and Oceania,Papua New Guinea,Meat,Offline,M,5/15/2015,940995585,6/4/2015,360,421.89,364.69,151880.40,131288.40,20592.00
3,Sub-Saharan Africa,Djibouti,Clothes,Offline,H,5/17/2017,880811536,7/2/2017,562,109.28,35.84,61415.36,20142.08,41273.28
4,Europe,Slovakia,Beverages,Offline,L,10/26/2016,174590194,12/4/2016,3973,47.45,31.79,188518.85,126301.67,62217.18


In [7]:
# Check duplicates count
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# Remove duplicates
df = df.drop_duplicates()

print("Duplicates removed. New shape:", df.shape)

Duplicate rows: 0
Duplicates removed. New shape: (949991, 14)


In [9]:
# Check missing values
missing_values = df.isnull().sum()
print(missing_values)

Region            0
Country           0
Item Type         0
Sales Channel     0
Order Priority    0
Order Date        0
Order ID          0
Ship Date         0
Units Sold        0
Unit Price        0
Unit Cost         0
Total Revenue     0
Total Cost        0
Total Profit      0
dtype: int64


In [11]:
# Drop rows with missing critical values
df = df.dropna(subset=["Order Date", "Sales Channel"])

# Fill numeric fields (if needed)
df["Units Sold"] = df["Units Sold"].fillna(df["Units Sold"].median())

In [12]:
# Clean text columns
df["Item Type"] = df["Item Type"].astype(str).str.strip().str.title()
df["Region"] = df["Region"].astype(str).str.strip().str.title()
df["Country"] = df["Country"].astype(str).str.strip().str.title()
df["Sales Channel"] = df["Sales Channel"].astype(str).str.strip().str.title()   

In [13]:
# Convert date columns
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

In [14]:
print(df["Order Date"].dtype)
print(df["Ship Date"].dtype)

datetime64[us]
datetime64[us]


In [16]:
# Convert numeric columns safely
numeric_columns = [
    "Units Sold",
    "Unit Price",
    "Unit Cost",
    "Total Revenue",
    "Total Cost",
    "Total Profit"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [17]:
df[numeric_columns].describe()

,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
count,949991.000000,949991.000000,949991.000000,9.499910e+05,9.499910e+05,9.499910e+05
mean,4999.040126,265.998343,187.501584,1.329458e+06,9.371750e+05,3.922827e+05
std,2885.526744,216.963625,175.626651,1.468399e+06,1.148812e+06,3.788344e+05
min,1.000000,9.330000,6.920000,9.330000e+00,6.920000e+00,2.410000e+00
25%,2502.000000,81.730000,35.840000,2.776774e+05,1.616846e+05,9.505707e+04
50%,4997.000000,154.060000,97.440000,7.847740e+05,4.669176e+05,2.811283e+05
75%,7497.000000,421.890000,263.330000,1.822334e+06,1.196607e+06,5.653648e+05
max,10000.000000,668.270000,524.960000,6.682700e+06,5.249600e+06,1.738700e+06


In [18]:
# If invalid is shown remove negative valuesmusing this method
df = df[df["Units Sold"] >= 0]
df = df[df["Total Revenue"] >= 0]

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 949991 entries, 0 to 999999
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Region          949991 non-null  str           
 1   Country         949991 non-null  str           
 2   Item Type       949991 non-null  str           
 3   Sales Channel   949991 non-null  str           
 4   Order Priority  949991 non-null  str           
 5   Order Date      949991 non-null  datetime64[us]
 6   Order ID        949991 non-null  int64         
 7   Ship Date       949991 non-null  datetime64[us]
 8   Units Sold      949991 non-null  int64         
 9   Unit Price      949991 non-null  float64       
 10  Unit Cost       949991 non-null  float64       
 11  Total Revenue   949991 non-null  float64       
 12  Total Cost      949991 non-null  float64       
 13  Total Profit    949991 non-null  float64       
dtypes: datetime64[us](2), float64(5), int64(2), str(5)
m

In [24]:
# Create the output directory if it doesn't exist
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "sales_records_clean.csv", index=False)